In [1]:
# %pip install python-dotenv
# %uv add dspy

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))


### check aicodetools library

In [3]:
import time
import dspy
from dspy.utils.callback import BaseCallback


class DelayAndLogCallback(BaseCallback):
    """Adds 5s delay and logs args/kwargs for each LM call."""

    def on_lm_start(self, *args, **kwargs):
        # print("\n🚀 [on_lm_start] LLM call starting...")
        # print(f"Args: {args}")"
        # print(f"Kwargs: {kwargs}")
        # print("⏳ Waiting 5 seconds before sending request...\n")"
        time.sleep(10)

    def on_lm_end(self, *args, **kwargs):
        # print("\n✅ [on_lm_end] LLM call completed.")
        # print(f"Args: {args}")
        # print(f"Kwargs: {kwargs}")
        # print("⏳ Waiting 5 seconds after receiving response...\n")
        # # time.sleep(2)
        time.sleep(1)

In [4]:

# from aicodetools import ClientManager 

# code_tool_manager = ClientManager(
#                 "super-bench:latest", base_log_dir="runs/super/"
#             )

# code_tool_client = code_tool_manager.get_client('initial')

In [5]:
import os
# os.environ['OPENAI_API_KEY'/] = input()
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [6]:
import dspy
lm = dspy.LM("azure/gpt-4.1", temperature=1.0, num_retries=0, callbacks=[DelayAndLogCallback()])
tlm = dspy.LM("azure/gpt-4.1",temperature=1.0)
dspy.configure(lm=lm)

In [7]:
# print(lm("Say this is a test!") ) # => ['This is a test!']
print(lm(messages=[{"role": "user", "content": "Say this is a test!"}]))  # => ['This is a test!']
print(tlm(messages=[{"role": "user", "content": "t : Say this is a test!"}]))  # => ['This is a test!']

['This is a test!']
['This is a test!']


## Load the benchmark and view one example from the benchmark

In [8]:
from gepa_artifact.benchmarks.super_bench.super_utils import FinishResponse
from gepa_artifact.benchmarks.super_bench import benchmark as sb_metas

In [9]:
bench = sb_metas[0].benchmark()

In [10]:
len(bench.train_set), len(bench.val_set), len(bench.test_set)

(9, 9, 27)

In [11]:
import pprint
pprint.pprint(bench.train_set[0])

Example({'instance_id': 'pie-perf', 'github_repo': 'https://github.com/madaan/pie-perf', 'git_commit': 'ee1989b66756470622e3b89c4aa031f083f57ef9', 'query': 'Evaluate the generations of my code improving model which are provided in https://drive.google.com/file/d/1izs1iF5cd_NAZsOaZvrrQF3NAsoP8lHf/view?usp=sharing (v1 vs v0). Once evaluated, report the result problem_id and input_acc for each problem of the dataset, as a json list of dictionaries structured as follows: [{"problem_id": "", "input_acc": 0.0}] (replace "" and 0.0 with the actual values).\n\nAdditional instructions:\n1. Set "num_trials": 2 in the evaluation configuration file to reduce computation time.\n2. Load only the first 10 rows of the dataset.\n\nGit repository: https://github.com/madaan/pie-perf', 'query_components': {'e2e_task': 'Evaluate the generations of my code improving model which are provided in https://drive.google.com/file/d/1izs1iF5cd_NAZsOaZvrrQF3NAsoP8lHf/view?usp=sharing (v1 vs v0).', 'scenario_task': '

## Load the program and display the program
The program is a 3-module system, each of which handles the urgency, sentiment and categories classification respectively

In [12]:
program = sb_metas[0].program[0]
program

react.react = Predict(StringSignature(query, github_repo, git_commit, trajectory -> next_thought, next_tool_name, next_tool_args
    instructions='Solve the question and provide the answer in the correct format.\n\nYou are an Agent. In each episode, you will be given the fields `query`, `github_repo`, `git_commit` as input. And you can see your past trajectory so far.\nYour goal is to use one or more of the supplied tools to collect any necessary information for producing `result`.\n\nTo do this, you will interleave next_thought, next_tool_name, and next_tool_args in each turn, and also when finishing the task.\nAfter each tool call, you receive a resulting observation, which gets appended to your trajectory.\n\nWhen writing next_thought, you may reason about the current situation and plan for future steps.\nWhen selecting the next_tool_name and its next_tool_args, the tool must be one of:\n\n(1) finish, whose description is <desc>Marks the task as complete. That is, signals that all i

### Make Sure docker is installed and running

## Define an evaluator and evaluate the base program

In [13]:
import dspy
evaluate = dspy.Evaluate(
    devset=bench.test_set,
    metric=sb_metas[0].metric,
    num_threads=1,
    display_table=True,
    display_progress=True,
    max_errors=100 * len(bench.test_set),
    provide_traceback = True,
    failure_score=0
)

## Load the GEPA Optimizer

In [14]:
# Import GEPA and define the optimizer
from gepa_artifact.gepa.gepa import GEPA
from gepa_artifact.utils.capture_stream_logger import Logger

import time

runs_dir = os.path.join(os.getcwd(), "runs", time.strftime("%Y-%m-%d_%H-%M-%S"))
os.makedirs(runs_dir, exist_ok=True)

gepa_logger = Logger(os.path.join(runs_dir, "run_log.txt"))

if sb_metas[0].feedback_fn_maps is None or sb_metas[0].feedback_fn_maps[0] is None:
    def feedback_func(predictor_output, predictor_inputs, module_inputs, module_outputs, captured_trace):
        pred = sb_metas[0].metric_with_feedback(module_inputs, module_outputs, None)
        return {
            "feedback_score": pred.score,
            "feedback_text": pred.feedback,
        }

    feedback_fn_map = {k:feedback_func for k, v in program.named_predictors()}
else:
    feedback_fn_map = sb_metas[0].feedback_fn_maps[0]

optimizer = GEPA(
    named_predictor_to_feedback_fn_map=feedback_fn_map,
    knowledgebase_qe=None,
    metric=sb_metas[0].metric,
    run_linearized_gepa=False,
    use_merge=True, 
    teacher_lm = tlm,
    set_for_merge_minibatch='val', 
    track_scores_on='val',
    max_metric_calls=100,
    run_dir=runs_dir,
    logger=gepa_logger,
    num_threads=2)

## Optimize the program with GEPA

In [15]:
x = sb_metas[0].program[0].get_lm()

print(x)

None


In [ ]:
optimized_program = optimizer.compile(
    sb_metas[0].program[0],
    trainset=bench.train_set,
    valset=bench.val_set,
)

ERROR:root:Failed to start Docker server: Port 18100 is already allocated


Available Tools :  4
Available Tools :  4
Prediction(
    trajectory={'thought_0': "I need to locate the quick start pipeline example/script for training and evaluating models in this repository, understand its usage (especially how to specify model, dataset, number of epochs, and sample subset), and check how metrics are reported. I'll start by searching for relevant scripts/notebooks in the repository root and documentation.", 'tool_name_0': 'run_command', 'tool_args_0': {'command': 'ls -l /workspace', 'timeout': 30, 'interactive': False}, 'observation_0': {'success': True, 'output': 'Exit code: 0\ntotal 0', 'message': 'executed in 0.01sec\nProcess finished and cleaned'}, 'thought_1': "The initial directory listing shows no files (likely inside /workspace is empty or the repo is in a subfolder). I'll check common subfolders for the main code or documentation, such as 'textbox', 'examples', or 'docs', by listing the contents in /workspace.", 'tool_name_1': 'run_command', 'tool_args_1'

2025/10/15 19:29:35 INFO dspy.evaluate.evaluate: Average Metric: 1.1666666666666667 / 9 (13.0%)


Prediction(
    trajectory={'thought_0': 'To begin, I need to understand the workflow for MNLI in this repo, including which scripts support LogME and classification, and how conversion to CSV-format works. I will start by listing the files in the repo root to locate relevant scripts and folders (such as main.py, classify.py, run_logme.sh, run_classification.sh, convert.py, etc.).', 'tool_name_0': 'run_command', 'tool_args_0': {'command': 'ls -la /workspace', 'timeout': 20}, 'observation_0': {'success': True, 'output': 'Exit code: 0\ntotal 8\ndrwxr-xr-x 2 root root 4096 Sep 18 09:56 .\ndrwxr-xr-x 1 root root 4096 Oct 15 19:07 ..', 'message': 'executed in 0.01sec\nProcess finished and cleaned'}, 'thought_1': "The listing suggests an empty workspace, which likely means the repository hasn't been cloned yet. To proceed, I need to clone the repository from the provided URL and checkout the specified commit so that I have access to the required scripts for LogME, classification, and convers

ValueError: Multiple LMs are being used in the module. There's no unique LM to return.

Prediction(
    trajectory={'thought_0': "To begin, I need to understand the folder structure and locate where the slot labelling model code, French data, and training scripts are. I'll start by listing the root directory to look for README, data, and relevant scripts.", 'tool_name_0': 'run_command', 'tool_args_0': {'command': 'ls -la /workspace', 'timeout': 20}, 'observation_0': {'success': True, 'output': 'Exit code: 0\ntotal 12\ndrwxr-xr-x 1 root root 4096 Oct 15 19:00 .\ndrwxr-xr-x 1 root root 4096 Oct 15 18:58 ..\ndrwxr-xr-x 6 root root 4096 Oct 15 19:02 multi3woz', 'message': 'executed in 0.01sec\nProcess finished and cleaned'}, 'thought_1': "The codebase is located under the /workspace/multi3woz directory. I need to look inside this directory for documentation (README), scripts, and the data directory to identify how to proceed with slot labelling for the French language. I'll list the contents of /workspace/multi3woz.", 'tool_name_1': 'run_command', 'tool_args_1': {'command': '

2025/10/15 19:44:41 ERROR dspy.utils.parallelizer: Error for Example({'instance_id': 'logme-nlp', 'github_repo': 'https://github.com/mainlp/logme-nlp', 'git_commit': '0046c725f2a3c1c8f0c8e77ef4603e57dcdae755', 'query': 'Calculate LogME for the MNLI dataset (using the `main.py` or `run_logme.sh` script) with the bert-base-uncased model. Then train a classifier for the same model and dataset (using the `classify.py` or `run_classification.sh` script). Report the LogME value, and the following classification results: macro-f1, micro-f1, and weighted-macro-f1 for both the matched and mismatched validation sets as a json structured as follows: {"LogMe": 0.0, "matched": {"macro-f1": 0.0, "micro-f1": 0.0, "weighted-macro-f1": 0.0}, "mismatched": {"macro-f1": 0.0, "micro-f1": 0.0, "weighted-macro-f1": 0.0}} (replace 0.0 with the actual values).\n\nAdditional instructions:\n1. Run for only 1 epoch.\n2. Run with seed 4012\n3. Use the `transformer+cls` embedding type and `first` pooling.\n4. Once

## Now, let's evaluate the optimized program

In [ ]:
evaluate(optimized_program)

GEPA was able to optimize the base program **from 57% score to 61% score** in just 9 iterations. With higher budget, the optimized program's score can go as high as **64%**.

### Let's print the prompts that GEPA discovered

In [ ]:
for name, pred in optimized_program.named_predictors():
    print("================================")
    print(f"Predictor: {name}")
    print("================================")
    print("Prompt:")
    print(pred.signature.instructions)
    print("*********************************")